In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from daemon_analysis_tools.io.csv_handler import load_and_process_csv
from daemon_analysis_tools.io.yaml_handler import save_answers_to_yaml, load_answers_from_yaml
from daemon_analysis_tools.processing.grouper import group_questions_by_journal
from daemon_analysis_tools.services.discrepancy_resolver import resolve_discrepancy

Load and process data:
- Group answers by publisher and journal, trying to uniform names written in slightly different ways.
- Store in a DataFrame

In [3]:
data = load_and_process_csv("../../data/raw/rdp.csv")

Get a `dict` labeled by publisher names of `dict`s labeled by journal names of `dict`s of `Question` instances. The `.answer` attribute contains the answers given by the respondents and the explanations text to motivate it.

In [4]:
question_metadata_file = "../../data/metadata/question_metadata.yaml"

grouped_questions = group_questions_by_journal(data, question_metadata_file)

## Resolve discrepancies

The `Question` class has a `.resolve_discrepancies` method which updates `Question.anwsers` with the correct answer.

For example, let's consider IOP's 2D Materials. Question 7 has discrepancies.

In [5]:
#print(list(grouped_questions.keys()))

for journal, data in grouped_questions["open journals"].items():
    print(journal)
    for question, answer in data.items():
        if answer.has_discrepancies():
            answer.print_qa()
    print("\n\n")

journal_of_open_source_software_(joss)
3. Data sharing requirements in RDP
  Resp. 0:
    Answer: Public data sharing of all data required.
    Explanation: “Code and associated data must be made openly available.”
  Resp. 1:
    Answer: Data sharing only with editors and referees (no public sharing)
    Explanation: "If any original data analysis results are provided within the submitted work, such results have to be entirely reproducible by reviewers, including accessing the data."
"Submissions are, by definition, contained within one or multiple repositories, which we require authors to archive before we accept the submission. Any data contained within the software is made available accordingly."
4. FAIR data sharing (see https://www.go-fair.org/fair-principles/ for a definition of FAIR)
  Resp. 0:
    Answer: Public data sharing of all data on a FAIR repository required.
    Explanation: “JOSS encourages the use of Zenodo and similar repositories with DOI assignment.”
  Resp. 1:
  

Inconsistencies can be removed manually, passing the index of the correct respondent.

In [6]:
for j in ["journal_of_open_source_software_(joss)"]:
    for i in [2, 3, 4, 7, 9, 16, 17, 19]:
        resolve_discrepancy(
            grouped_questions["open journals"][j][i],
            correct_answer=0,
            discrepancy_reason="Text not found",
        )
    
    for i in [11, 12]:
        resolve_discrepancy(
            grouped_questions["open journals"][j][i],
            correct_answer=0,
            discrepancy_reason="Formatting",
        )
    
    for i in [14]:
        resolve_discrepancy(
            grouped_questions["open journals"][j][i],
            correct_answer=0,
            discrepancy_reason="Language understanding",
        )

In [7]:
for journal, data in grouped_questions["open journals"].items():
    print("#############################################################")
    print(journal)
    for question, answer in data.items():
        if answer.has_discrepancies() and answer.correct_answer is None:
            answer.print_qa()

#############################################################
journal_of_open_source_software_(joss)


In [8]:
save_answers_to_yaml(
    grouped_questions,
    parent_folder="../../data/processed/all_answers",
    save_only=["open journals"],
)

After doing this, the `.get_final_answer()` method returns the correct answer.